# Mixed-Effects Models

In a typical approach, each metric is averaged across all trials in a session, reducing each person to a single number per metric. This loses the trial-by-trial structure of the data.

Here we keep every individual trial as a separate row and use mixed-effects models to ask two questions:

1. **Is depression associated with attention metrics?** (do people with higher PHQ-9 or BDI scores look at images differently?)
2. **Does depression change how attention evolves across trials?** (do depressed people's viewing patterns shift differently over the course of the test compared to non-depressed people?)

The model accounts for the fact that trials are nested within people — each person has their own baseline and their own natural drift over time. This prevents individual quirks from being mistaken for depression effects.

We test the following attention metrics:

- `fixation_bias` — proportional difference in looking time between negative and positive images (-1 to +1)
- `dwell_time_ms_negative` — total time spent looking at the negative image (ms)
- `dwell_time_ms_positive` — total time spent looking at the positive image (ms)
- `dwell_time_ms_neutral` — total time spent looking at the neutral image (ms)
- `fixation_proportion_negative` — share of total looking time that went to the negative image
- `fixation_proportion_positive` — share of total looking time that went to the positive image
- `first_fixation_duration_ms` — how long the very first look at any image lasted (ms)
- `dwell_time_500ms_negative` — time spent on the negative image within the first 500ms of the trial
- `dwell_time_500ms_positive` — time spent on the positive image within the first 500ms of the trial
- `revisit_count_negative` — how many times gaze returned to the negative image after leaving it
- `revisit_count_positive` — how many times gaze returned to the positive image after leaving it
- `blink_rate_per_min` — number of blinks per minute during the trial
- `scanpath_length` — total distance the eyes traveled across the screen

For each metric, we fit the following model:

`metric ~ depression_score * trial_position`, with random effects per user.

What this means in practice:

- **Depression score effect**: does the metric differ between people with higher and lower depression scores? For example, do people with higher PHQ-9 spend more time looking at negative images?
- **Trial position effect**: does the metric change as the test goes on? For example, does everyone start looking at images less over time regardless of their depression level?
- **Interaction (depression × trial position)**: does the way the metric changes over trials depend on depression level? For example, do depressed people lose interest faster, or do they maintain attention on negative images longer than non-depressed people?

The depression scores are standardized (converted to z-scores, where 0 = average, +1 = one standard deviation above average) so that PHQ-9 and BDI results are comparable. Trial position is normalized to a 0-1 range (0 = first trial, 1 = last trial) for numerical stability.

The model also includes two random effects that control for individual differences:

- **Random intercept per user**: each person has their own baseline level of attention. Some people just look at everything longer than others, regardless of depression. The random intercept absorbs this so it doesn't contaminate the depression effect.
- **Random slope per user**: each person's attention drifts over trials in their own way. Some people stay focused, others lose concentration quickly. The random slope absorbs this so the interaction test only captures drift that is systematically related to depression, not individual quirks.

If the random slope causes the model to fail (not enough data to estimate it reliably), the model falls back to random intercept only.

We run the analysis separately for PHQ-9 and BDI scores, then compare the results.

In [0]:
%pip install statsmodels -q

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np

from src.mixed_effects import fit_all_metrics, build_summary_df, plot_trajectories, compare_scales

## 1. Build trial-level dataset

In [0]:
scene_metrics = spark.table("anima.scene_metrics")
df_forms = spark.table("anima.forms")

df_stimulus = scene_metrics.filter(F.col("scene_type") == "stimulus")

w = Window.partitionBy("session_id").orderBy("scene_index")
numbered = df_stimulus.withColumn("trial_num", F.row_number().over(w))

METRICS = [
    "fixation_bias",
    "dwell_time_ms_negative", "dwell_time_ms_positive", "dwell_time_ms_neutral",
    "fixation_proportion_negative", "fixation_proportion_positive",
    "first_fixation_duration_ms",
    "dwell_time_500ms_negative", "dwell_time_500ms_positive",
    "revisit_count_negative", "revisit_count_positive",
    "blink_rate_per_min",
    "scanpath_length",
]

cols = ["session_id", "scene_index", "trial_num"] + METRICS

df = numbered.select(*cols).join(
    df_forms.select("session_id", "uid", "phq9_score", "bdi_score"),
    on="session_id", how="inner",
).toPandas()

df["trial_norm"] = df.groupby("session_id")["trial_num"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() > x.min() else 0
)
df["phq9_z"] = (df["phq9_score"] - df["phq9_score"].mean()) / df["phq9_score"].std()
df["bdi_z"] = (df["bdi_score"] - df["bdi_score"].mean()) / df["bdi_score"].std()

print(f"Rows: {len(df)}") 
print(f"Sessions: {df['session_id'].nunique()}")
print(f"Users: {df['uid'].nunique()}")
print(f"PHQ-9: {df['phq9_score'].min()}-{df['phq9_score'].max()}, mean={df['phq9_score'].mean():.1f}")
print(f"BDI: {df['bdi_score'].min()}-{df['bdi_score'].max()}, mean={df['bdi_score'].mean():.1f}")

## 2. PHQ-9

### 2.1 Fit models

In [0]:
phq_results = fit_all_metrics(df, METRICS, "phq9_z")

### 2.2 Summary

In [0]:
phq_summary = build_summary_df(phq_results, "phq9_z")
print(phq_summary.to_string(index=False))

### 2.3 Detailed results for key metrics

In [0]:
KEY_METRICS = ["fixation_bias", "dwell_time_ms_negative", "dwell_time_ms_positive",
               "fixation_proportion_negative", "fixation_proportion_positive"]

for i, metric in enumerate(KEY_METRICS):
    if metric in phq_results:
        print(f"\n{i+1}. {metric} (PHQ-9)")
        print(phq_results[metric].summary())

### 2.4 Trajectories

In [0]:
plot_trajectories(df, phq_summary, "phq9_z", "PHQ-9", "phq9_score", METRICS)

## 3. BDI-II

### 3.1 Fit models

In [0]:
bdi_results = fit_all_metrics(df, METRICS, "bdi_z")

### 3.2 Summary

In [0]:
bdi_summary = build_summary_df(bdi_results, "bdi_z")
print(bdi_summary.to_string(index=False))

### 3.3 Detailed results for key metrics

In [0]:
for i, metric in enumerate(KEY_METRICS):
    if metric in bdi_results:
        print(f"\n{i+1}. {metric} (BDI)")
        print(bdi_results[metric].summary())

### 3.4 Trajectories

In [0]:
plot_trajectories(df, bdi_summary, "bdi_z", "BDI", "bdi_score", METRICS)

## 4. Comparison

In [0]:
compare_scales(phq_summary, bdi_summary)